# Driver monitoring: video demonstration

Run the **saved ResNet-18 distraction model and MobileNet eye-state model together** on a short local video. This creates a playable annotated preview, frame predictions, sustained-alert events and processing-time measurements. **No training or accuracy scoring happens here.**

Use the same **local project `.venv` kernel** as the testing notebook. No Kaggle session or data upload is needed. The default is the first 250 frames (10 seconds) of an existing NITYMED development clip. DMD is excluded. The clip is recorded as development footage and must not later be presented as an unseen test sample; participant identity still needs review.

Eye predictions cover open/closed eyes only. Yawning, nodding and gaze are not detected by this notebook. The two existing models remain separate; this does not establish the proposed shared multi-task model or video accuracy.


## 1. Choose inspection or demonstration

First keep `RUN_DEMO = False` and click **Run All**. After Step 2 says READY, set it to `True` and run once. The default sample is already local. To use your own development clip later, put it in `data/demo/` and enter its project-relative path in `SOURCE_VIDEO`.


In [ ]:
from pathlib import Path
import json
import sys

RUN_DEMO = False
MAX_FRAMES = 250
SOURCE_VIDEO = ""  # Blank uses the supplied NITYMED development preview.

assert not Path("/kaggle/working").is_dir(), "Select the local project .venv kernel."
REPO = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "configs/video_demo.json").is_file()), None
)
assert REPO is not None, "Open this notebook inside the project folder."
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
OUTPUT = REPO / "results/video_demo/local_preview"
print("Python:", sys.executable)
print("Video demo enabled:", RUN_DEMO)
print("Outputs:", OUTPUT)


## 2. Verify inputs

This checks both saved checkpoints, the face-landmark model, FFmpeg, and video metadata. It does not predict frames or change model weights. Both models use their own saved preprocessing; eye crops come from face landmarks, and distraction uses the entire frame.


In [ ]:
from src.demo.workflow import inspect_demo

plan = inspect_demo(REPO, SOURCE_VIDEO or None)
print("READY:", plan["ready"])
print("Distraction model:", plan["distraction_model"])
print("Eye model: MobileNetV3-Small, open/closed eyes")
print("Source:", plan["source"])
print("Video:", plan["source_dimensions"], "| FPS:", plan["source_fps"])
print("Frames to process:", min(MAX_FRAMES, plan["source_frames"]))
print("Alert demonstration settings:", plan["settings"])
print("These alert settings are not validated false-alarm operating points.")


## 3. Process the preview

When enabled, progress prints after the first frame and every 50 frames. Let it finish. Missing eye crops are logged as Unknown; the full-frame distraction model can still run. The default sustained-alert rule requires the same confident non-safe/closed-eye prediction for one second of source-video time.

Completed output is preserved on a repeated Run All. An incomplete output folder is also preserved; share its error with the coding agent rather than deleting it or overwriting a previous run.


In [ ]:
from src.demo.workflow import run_demo

if not RUN_DEMO:
    print("Inspection only. No video inference or training performed.")
elif (OUTPUT / "summary.json").is_file():
    existing = json.loads((OUTPUT / "input_provenance.json").read_text(encoding="utf-8"))
    for key in ("source_sha256", "distraction_checkpoint_sha256", "eye_checkpoint_sha256", "face_model_sha256"):
        assert existing[key] == plan[key], "Existing preview uses different inputs; preserve it."
    assert existing["settings"] == plan["settings"], "Existing preview uses different settings; preserve it."
    saved = json.loads((OUTPUT / "summary.json").read_text(encoding="utf-8"))
    assert saved["processed_frames"] == min(MAX_FRAMES, plan["source_frames"]), "Existing preview uses a different frame limit; preserve it."
    print("Completed preview already exists; preserving it. Watch it in Step 4.")
else:
    summary = run_demo(REPO, OUTPUT, MAX_FRAMES, SOURCE_VIDEO or None)


## 4. Watch and inspect

The overlay shows predictions and whether a sustained demonstration alert is active. These are model outputs, **not ground-truth labels**. Coverage is the number of frames where an eye crop was available, not eye-state accuracy. Counts of predicted alerts are not false-alarm rates.

The timing report measures local decode/landmark/classifier work after model loading. Video output is saved at the source FPS even if processing takes longer; smooth playback does not prove real-time performance. NITYMED has a different camera view from State Farm, so distraction predictions may be unreliable on this preview.


In [ ]:
from IPython.display import Video, display

if not (OUTPUT / "summary.json").is_file():
    print("No completed video demo yet. Enable RUN_DEMO after READY.")
else:
    summary = json.loads((OUTPUT / "summary.json").read_text(encoding="utf-8"))
    display(Video(str(OUTPUT / "annotated_preview.mp4"), embed=True, html_attributes="controls width=900"))
    print("Processed frames:", summary["processed_frames"])
    print(f"Eye-crop availability: {summary['eye_coverage']:.1%}")
    print("Predicted alert starts:", summary["alert_starts"])
    print(f"Processing time: {summary['processing_seconds']:.2f} seconds")
    timing = summary["decode_and_inference_ms"]
    print(f"Decode + inference: median {timing['median']:.1f} ms/frame; p95 {timing['p95']:.1f} ms/frame")
    print("Accuracy scoring:", summary["scoring_performed"])
    print("Training performed:", summary["training_performed"])


## 5. Save the notebook and report back

Press **Ctrl+S**, then tell the coding agent the video demo is complete. All files already live locally. The agent can inspect the logs and recording before preparing labeled external evaluation. No new training is required to run this demonstration.


In [ ]:
if (OUTPUT / "summary.json").is_file():
    print("Saved files:")
    for path in sorted(OUTPUT.iterdir()):
        print(" ", path.name)
    events = [json.loads(line) for line in (OUTPUT / "events.jsonl").read_text(encoding="utf-8").splitlines()]
    print("Alert transitions:")
    for event in events:
        if event["event"] in ("start", "end"):
            print(event)
    print("No event transitions can also be a valid demo outcome; there is no labeled score here.")
else:
    print("No demonstration output to inspect yet.")
